# Handling Imbalanced Data

Class imbalance occurs when one class has significantly more samples than others. This is common in real-world datasets and can hurt model performance, as algorithms tend to favor the majority class.

## Why Handle Imbalance?
- Prevent biased predictions
- Improve minority class detection
- Better use of training data
- More realistic model performance

## Common Imbalance Ratios:
- Mild: 1:2 to 1:10
- Moderate: 1:10 to 1:100
- Severe: > 1:100

## Techniques:
1. **Over-sampling** (Increase minority)
2. **Under-sampling** (Decrease majority)
3. **Synthetic Data** (SMOTE)
4. **Cost-sensitive Learning** (Weight adjustment)
5. **Ensemble Methods** (Balanced ensembles)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt

# Create imbalanced dataset
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    weights=[0.95, 0.05],  # 95-5 imbalance
    random_state=42
)

df = pd.DataFrame(X, columns=[f'Feature_{i}' for i in range(10)])
df['Target'] = y

print("Original Dataset:")
print(f"Total samples: {len(df)}")
print("\nClass distribution:")
print(df['Target'].value_counts())
print("\nClass distribution (%):")
print(df['Target'].value_counts(normalize=True) * 100)

## 1. Random Over-sampling

Duplicate samples from minority class randomly.

In [ ]:
# Random Over-sampling
minority_count = (df['Target'] == 1).sum()
majority_count = (df['Target'] == 0).sum()

minority_df = df[df['Target'] == 1]
majority_df = df[df['Target'] == 0]

# Oversample minority class
minority_oversampled = minority_df.sample(n=majority_count, replace=True, random_state=42)
df_oversampled = pd.concat([majority_df, minority_oversampled])

print("After Random Over-sampling:")
print(f"Total samples: {len(df_oversampled)}")
print("\nClass distribution:")
print(df_oversampled['Target'].value_counts())
print("\nClass distribution (%):")
print(df_oversampled['Target'].value_counts(normalize=True) * 100)

## 2. Random Under-sampling

Randomly remove samples from majority class.

In [ ]:
# Random Under-sampling
# Undersample majority class
majority_undersampled = majority_df.sample(n=minority_count, random_state=42)
df_undersampled = pd.concat([minority_df, majority_undersampled])

print("After Random Under-sampling:")
print(f"Total samples: {len(df_undersampled)}")
print("\nClass distribution:")
print(df_undersampled['Target'].value_counts())
print("\nClass distribution (%):")
print(df_undersampled['Target'].value_counts(normalize=True) * 100)

## 3. SMOTE (Synthetic Minority Over-sampling Technique)

Creates synthetic samples for minority class using K-Nearest Neighbors.

In [ ]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(df.drop('Target', axis=1), df['Target'])

df_smote = pd.DataFrame(X_smote)
df_smote['Target'] = y_smote

print("After SMOTE:")
print(f"Total samples: {len(df_smote)}")
print("\nClass distribution:")
print(df_smote['Target'].value_counts())
print("\nClass distribution (%):")
print(df_smote['Target'].value_counts(normalize=True) * 100)

## 4. Class Weights

Adjust model weights to penalize misclassification of minority class.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression

# Calculate class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
class_weight_dict = dict(enumerate(class_weights))

print("Class Weights (for use in model):")
print(class_weight_dict)
print(f"\nMinority class weight: {class_weight_dict[1]:.2f}")
print(f"Majority class weight: {class_weight_dict[0]:.2f}")

# Example: Using class weights in Logistic Regression
model = LogisticRegression(class_weight='balanced', random_state=42)
print("\nModel can use these weights directly with: class_weight='balanced'")